# Carga de Datos

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split 
from plotly import express as px
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE

In [ ]:
project_root = Path().resolve().parents[1]

data = project_root/ "data" / "processed" / "games_info_prices.parquet"
pd.set_option('display.max_columns', None)
df = pd.read_parquet(data)
df.head(2)

# Procesado para Modelo

##### Drop

In [ ]:
df.dtypes

In [ ]:
df.drop(columns=['id','name'], inplace=True, errors='ignore')
print(df.dtypes)
df.head(2)

##### Transformacion Release Year

In [ ]:

le = LabelEncoder()
le.fit(df['release_year'])
encoding = le.transform(df['release_year'])

df['release_year'] = pd.Series(encoding)
df

#### Normalización

In [ ]:
df.describe()

In [ ]:

columns_to_normalize = ['description_len', 'num_languages', 'total_games_by_publisher', 'total_games_by_developer']

scaler = StandardScaler()
df[columns_to_normalize] = scaler.fit_transform(df[columns_to_normalize])
df

In [ ]:
df.describe()

# Modelo KNN

In [ ]:
y = df['price_range']
X = df.drop(columns=['price_range'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42)    

In [ ]:

knn_scores = []
k_values = range(1,50)
for k_value in k_values:
    print(k_value, end=" ")
    knn = KNeighborsClassifier(n_neighbors=k_value)

    scores = cross_val_score(knn, X_train, y_train, cv=5)
    knn_scores.append(scores.mean())

best_k = k_values[np.argmax(knn_scores)]
print("\n")
print(knn_scores)
print(best_k, knn_scores[best_k])

In [ ]:

fig = px.line(x= np.arange(1,50,1), y=knn_scores)
fig.show()

In [ ]:

knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(X_train,y_train)
y_pred = knn.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
print(precision)
print(recall)
print(f1)
conf_matrix = confusion_matrix(y_test, y_pred)
print(conf_matrix)

In [ ]:
df['price_range'].unique()

In [ ]:
plt.figure(figsize=(6,5))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=df['price_range'].unique(), yticklabels=df['price_range'].unique())
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.title(f'Matriz de confusión (k={best_k})')
plt.show()

# SMOTE

In [ ]:
y_train.value_counts()

In [ ]:
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train, y_train)
y_res.value_counts()

In [ ]:
knn_scores = []
for k_value in range(1,50):

    print(k_value, end=" ")
    knn = KNeighborsClassifier(n_neighbors=k_value)

    scores = cross_val_score(knn, X_res, y_res, cv=5)
    knn_scores.append(scores.mean())


best_k = range(1,50)[np.argmax(knn_scores)]
print("\n")
print(knn_scores)
print(best_k, knn_scores[best_k])

In [ ]:

knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(X_res,y_res)
y_pred = knn.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
print(precision)
print(recall)
print(f1)
conf_matrix = confusion_matrix(y_test, y_pred)
print(conf_matrix)